# 12 — MS-TCN Baseline on Breakfast ProcedureVRL Features

This notebook trains and evaluates a visual-only MS-TCN-style TAS model on the full Breakfast split-1 ProcedureVRL feature dataset produced by notebook 11.

Input dataset:

```text
/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format
```

Expected structure:

```text
features/*.npy       # shape [9871, 16]
groundTruth/*.txt    # length 16
splits/*.bundle
mapping.txt
```

Important limitation:

These are **coarse clip-level ProcedureVRL features**, not dense frame-level I3D features.  
So this run is a **pipeline validation and ProcedureVRL-feature baseline**, not yet a direct paper-level comparison to frame-level I3D MS-TCN results.

## 1. Mount Drive and imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import os
import sys
import json
import time
import random
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Mounted at /content/drive
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


## 2. Paths and configuration

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")

PROC_RUN_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "breakfast"
    / "procedurevrl"
    / "runs"
    / "procedurevrl_breakfast_full_split1_split1_views16"
)

DATA_ROOT = PROC_RUN_ROOT / "mstcn_format"
FEATURE_DIR = DATA_ROOT / "features"
GT_DIR = DATA_ROOT / "groundTruth"
SPLIT_DIR = DATA_ROOT / "splits"
MAPPING_PATH = DATA_ROOT / "mapping.txt"

RUN_NAME = "mstcn_on_procedurevrl_breakfast_full_split1"
OUT_ROOT = DRIVE_ROOT / "text_assisted_tas" / "breakfast" / "procedurevrl" / "mstcn_runs" / RUN_NAME
PRED_DIR = OUT_ROOT / "predictions"
CKPT_DIR = OUT_ROOT / "checkpoints"
for p in [OUT_ROOT, PRED_DIR, CKPT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SPLIT_ID = 1
SEED = 42

# Start with a moderate setup
NUM_EPOCHS = 80
BATCH_SIZE = 32
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4

# Coarse features have T=16, so a very deep/dilated MS-TCN is unnecessary.
NUM_STAGES = 3
NUM_LAYERS = 4
NUM_F_MAPS = 64
DROPOUT = 0.5

# Temporal smoothing loss, as in MS-TCN-style training.
LAMBDA_SMOOTH = 0.15
MAX_SMOOTH_CLAMP = 16.0

device = "cuda" if torch.cuda.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

for p in [DATA_ROOT, FEATURE_DIR, GT_DIR, SPLIT_DIR, MAPPING_PATH]:
    assert p.exists(), f"Missing path: {p}"

print("DATA_ROOT:", DATA_ROOT)
print("OUT_ROOT:", OUT_ROOT)
print("device:", device)

DATA_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/mstcn_runs/mstcn_on_procedurevrl_breakfast_full_split1
device: cuda


## 3. Load mapping and split files

In [ ]:
def read_lines(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]

def load_mapping(mapping_path):
    id_to_label = {}
    label_to_id = {}
    for line in read_lines(mapping_path):
        idx, label = line.split()[:2]
        idx = int(idx)
        id_to_label[idx] = label
        label_to_id[label] = idx
    return id_to_label, label_to_id

def load_split_ids(path):
    return [Path(x).stem for x in read_lines(path)]

id_to_label, label_to_id = load_mapping(MAPPING_PATH)
num_classes = len(id_to_label)

train_ids = load_split_ids(SPLIT_DIR / f"train.split{SPLIT_ID}.bundle")
test_ids = load_split_ids(SPLIT_DIR / f"test.split{SPLIT_ID}.bundle")

print("num_classes:", num_classes)
print("train videos:", len(train_ids))
print("test videos:", len(test_ids))
print("first train ids:", train_ids[:5])
print("first labels:", list(label_to_id.items())[:10])

num_classes: 48
train videos: 1460
test videos: 252
first train ids: ['P16_cam01_P16_cereals', 'P16_cam01_P16_friedegg', 'P16_cam01_P16_juice', 'P16_cam01_P16_milk', 'P16_cam01_P16_pancake']
first labels: [('SIL', 0), ('pour_cereals', 1), ('pour_milk', 2), ('stir_cereals', 3), ('take_bowl', 4), ('pour_coffee', 5), ('take_cup', 6), ('spoon_sugar', 7), ('stir_coffee', 8), ('pour_sugar', 9)]


## 4. Validate feature/label dataset

In [ ]:
validation_rows = []

for video_id in train_ids + test_ids:
    feat_path = FEATURE_DIR / f"{video_id}.npy"
    gt_path = GT_DIR / f"{video_id}.txt"

    assert feat_path.exists(), feat_path
    assert gt_path.exists(), gt_path

    feat = np.load(feat_path, mmap_mode="r")
    labels = read_lines(gt_path)

    unknown_labels = sorted(set(labels) - set(label_to_id))
    validation_rows.append({
        "video_id": video_id,
        "split": "train" if video_id in set(train_ids) else "test",
        "feature_shape": list(feat.shape),
        "feature_dim": int(feat.shape[0]),
        "feature_len": int(feat.shape[1]),
        "gt_len": len(labels),
        "length_match": int(feat.shape[1]) == len(labels),
        "unknown_labels": ",".join(unknown_labels),
    })

df_val = pd.DataFrame(validation_rows)
display(df_val.head())

print("videos:", len(df_val))
print("length mismatches:", int((~df_val["length_match"]).sum()))
print("unknown-label rows:", int((df_val["unknown_labels"] != "").sum()))
print("feature_dim unique:", sorted(df_val["feature_dim"].unique().tolist()))
print("feature_len unique:", sorted(df_val["feature_len"].unique().tolist()))

assert df_val["length_match"].all()
assert (df_val["unknown_labels"] == "").all()

feature_dim = int(df_val["feature_dim"].iloc[0])
feature_len = int(df_val["feature_len"].iloc[0])

print("feature_dim:", feature_dim)
print("feature_len:", feature_len)

,video_id,split,feature_shape,feature_dim,feature_len,gt_len,length_match,unknown_labels
0,P16_cam01_P16_cereals,train,"[9871, 16]",9871,16,16,True,
1,P16_cam01_P16_friedegg,train,"[9871, 16]",9871,16,16,True,
2,P16_cam01_P16_juice,train,"[9871, 16]",9871,16,16,True,
3,P16_cam01_P16_milk,train,"[9871, 16]",9871,16,16,True,
4,P16_cam01_P16_pancake,train,"[9871, 16]",9871,16,16,True,


videos: 1712
length mismatches: 0
unknown-label rows: 0
feature_dim unique: [9871]
feature_len unique: [16]
feature_dim: 9871
feature_len: 16


## 5. Dataset and DataLoader

In [ ]:
class ProcedureVRLBreakfastDataset(Dataset):
    def __init__(self, video_ids, feature_dir, gt_dir, label_to_id):
        self.video_ids = list(video_ids)
        self.feature_dir = Path(feature_dir)
        self.gt_dir = Path(gt_dir)
        self.label_to_id = dict(label_to_id)

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        video_id = self.video_ids[idx]

        feat = np.load(self.feature_dir / f"{video_id}.npy").astype(np.float32)  # [D, T]
        labels = read_lines(self.gt_dir / f"{video_id}.txt")
        y = np.array([self.label_to_id[x] for x in labels], dtype=np.int64)      # [T]

        if feat.shape[1] != len(y):
            raise ValueError(f"Length mismatch for {video_id}: feat {feat.shape}, labels {len(y)}")

        return {
            "video_id": video_id,
            "features": torch.from_numpy(feat),
            "labels": torch.from_numpy(y),
        }

def collate_batch(batch):
    # Here all T=16, but keep padding logic for safety.
    video_ids = [b["video_id"] for b in batch]
    lengths = [b["features"].shape[1] for b in batch]
    max_len = max(lengths)
    dim = batch[0]["features"].shape[0]

    x = torch.zeros(len(batch), dim, max_len, dtype=torch.float32)
    y = torch.full((len(batch), max_len), -100, dtype=torch.long)
    mask = torch.zeros(len(batch), max_len, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["features"].shape[1]
        x[i, :, :T] = b["features"]
        y[i, :T] = b["labels"]
        mask[i, :T] = True

    return video_ids, x, y, mask

train_ds = ProcedureVRLBreakfastDataset(train_ids, FEATURE_DIR, GT_DIR, label_to_id)
test_ds = ProcedureVRLBreakfastDataset(test_ids, FEATURE_DIR, GT_DIR, label_to_id)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_batch,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_batch,
    pin_memory=torch.cuda.is_available(),
)

video_ids, x, y, mask = next(iter(train_loader))
print("batch x:", x.shape)
print("batch y:", y.shape)
print("batch mask:", mask.shape)
print("first video:", video_ids[0])

batch x: torch.Size([32, 9871, 16])
batch y: torch.Size([32, 16])
batch mask: torch.Size([32, 16])
first video: P48_cam01_P48_juice


## 6. MS-TCN-style model

In [ ]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, channels, dropout):
        super().__init__()
        self.conv_dilated = nn.Conv1d(channels, channels, kernel_size=3, padding=dilation, dilation=dilation)
        self.conv_1x1 = nn.Conv1d(channels, channels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = F.relu(self.conv_dilated(x))
        out = self.conv_1x1(out)
        out = self.dropout(out)
        return x + out

class SingleStageModel(nn.Module):
    def __init__(self, num_layers, in_dim, num_f_maps, num_classes, dropout):
        super().__init__()
        self.conv_in = nn.Conv1d(in_dim, num_f_maps, kernel_size=1)
        self.layers = nn.ModuleList([
            DilatedResidualLayer(2 ** i, num_f_maps, dropout)
            for i in range(num_layers)
        ])
        self.conv_out = nn.Conv1d(num_f_maps, num_classes, kernel_size=1)

    def forward(self, x):
        out = self.conv_in(x)
        for layer in self.layers:
            out = layer(out)
        return self.conv_out(out)

class MultiStageModel(nn.Module):
    def __init__(self, num_stages, num_layers, in_dim, num_f_maps, num_classes, dropout):
        super().__init__()
        self.stage1 = SingleStageModel(num_layers, in_dim, num_f_maps, num_classes, dropout)
        self.stages = nn.ModuleList([
            SingleStageModel(num_layers, num_classes, num_f_maps, num_classes, dropout)
            for _ in range(num_stages - 1)
        ])

    def forward(self, x):
        outputs = []
        out = self.stage1(x)
        outputs.append(out)
        for stage in self.stages:
            out = stage(F.softmax(out, dim=1))
            outputs.append(out)
        return outputs  # list of [B, C, T]

model = MultiStageModel(
    num_stages=NUM_STAGES,
    num_layers=NUM_LAYERS,
    in_dim=feature_dim,
    num_f_maps=NUM_F_MAPS,
    num_classes=num_classes,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print("parameters:", n_params)

MultiStageModel(
  (stage1): SingleStageModel(
    (conv_in): Conv1d(9871, 64, kernel_size=(1,), stride=(1,))
    (layers): ModuleList(
      (0): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (1): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(2,), dilation=(2,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (2): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(4,), dilation=(4,))
        (conv_1x1): Conv1d(64, 64, kernel_size=(1,), stride=(1,))
        (dropout): Dropout(p=0.5, inplace=False)
      )
      (3): DilatedResidualLayer(
        (conv_dilated): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(8,

## 7. Loss, metrics, and evaluation functions

In [ ]:
ce_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)

def mstcn_loss(outputs, targets):
    total = 0.0
    for out in outputs:
        ce = ce_loss_fn(out.transpose(1, 2).reshape(-1, out.shape[1]), targets.reshape(-1))

        # Temporal smoothing on log-probabilities.
        logp = F.log_softmax(out, dim=1)
        smooth = torch.mean(torch.clamp((logp[:, :, 1:] - logp.detach()[:, :, :-1]) ** 2, max=MAX_SMOOTH_CLAMP))

        total = total + ce + LAMBDA_SMOOTH * smooth
    return total

def frame_accuracy_from_logits(logits, targets, mask):
    pred = logits.argmax(dim=1)  # [B, T]
    correct = ((pred == targets) & mask).sum().item()
    total = mask.sum().item()
    return correct / max(total, 1)

def collapse_segments(labels, background=None):
    seg_labels = []
    seg_starts = []
    seg_ends = []

    prev = None
    start = 0

    for i, lab in enumerate(labels):
        if lab != prev:
            if prev is not None and prev != background:
                seg_labels.append(prev)
                seg_starts.append(start)
                seg_ends.append(i)
            prev = lab
            start = i

    if prev is not None and prev != background:
        seg_labels.append(prev)
        seg_starts.append(start)
        seg_ends.append(len(labels))

    return seg_labels, seg_starts, seg_ends

def edit_score(recognized, ground_truth, background=None):
    # Standard Levenshtein on collapsed segment labels.
    P, _, _ = collapse_segments(recognized, background)
    Y, _, _ = collapse_segments(ground_truth, background)

    m, n = len(P), len(Y)
    if n == 0:
        return 100.0 if m == 0 else 0.0

    D = np.zeros((m + 1, n + 1), dtype=np.float32)
    for i in range(m + 1):
        D[i, 0] = i
    for j in range(n + 1):
        D[0, j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if P[i - 1] == Y[j - 1]:
                D[i, j] = D[i - 1, j - 1]
            else:
                D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + 1)

    return (1.0 - D[m, n] / max(m, n)) * 100.0

def f_score(recognized, ground_truth, overlap, background=None):
    p_label, p_start, p_end = collapse_segments(recognized, background)
    y_label, y_start, y_end = collapse_segments(ground_truth, background)

    tp = 0
    fp = 0
    hits = np.zeros(len(y_label), dtype=np.float32)

    for j in range(len(p_label)):
        intersection = np.minimum(p_end[j], y_end) - np.maximum(p_start[j], y_start)
        union = np.maximum(p_end[j], y_end) - np.minimum(p_start[j], y_start)
        IoU = (intersection / np.maximum(union, 1e-8)) * ([p_label[j] == y for y in y_label])

        idx = np.array(IoU).argmax() if len(IoU) else -1
        if len(IoU) and IoU[idx] >= overlap and not hits[idx]:
            tp += 1
            hits[idx] = 1
        else:
            fp += 1

    fn = len(y_label) - hits.sum()
    precision = tp / float(tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / float(tp + fn) if (tp + fn) > 0 else 0.0

    if precision + recall == 0:
        return 0.0

    return 2.0 * (precision * recall) / (precision + recall) * 100.0

@torch.no_grad()
def evaluate_model(model, loader, save_predictions=False, pred_dir=None):
    model.eval()

    total_correct = 0
    total_frames = 0
    edit_scores = []
    f1_10 = []
    f1_25 = []
    f1_50 = []

    if save_predictions:
        pred_dir = Path(pred_dir)
        pred_dir.mkdir(parents=True, exist_ok=True)

    for video_ids, x, y, mask in loader:
        x = x.to(device)
        y = y.to(device)
        mask = mask.to(device)

        outputs = model(x)
        logits = outputs[-1]
        pred = logits.argmax(dim=1)  # [B, T]

        total_correct += ((pred == y) & mask).sum().item()
        total_frames += mask.sum().item()

        pred_np = pred.cpu().numpy()
        y_np = y.cpu().numpy()
        mask_np = mask.cpu().numpy()

        for i, vid in enumerate(video_ids):
            T = int(mask_np[i].sum())
            pred_ids = pred_np[i, :T].tolist()
            true_ids = y_np[i, :T].tolist()

            pred_labels = [id_to_label[int(k)] for k in pred_ids]
            true_labels = [id_to_label[int(k)] for k in true_ids]

            edit_scores.append(edit_score(pred_labels, true_labels))
            f1_10.append(f_score(pred_labels, true_labels, 0.10))
            f1_25.append(f_score(pred_labels, true_labels, 0.25))
            f1_50.append(f_score(pred_labels, true_labels, 0.50))

            if save_predictions:
                (pred_dir / f"{vid}.txt").write_text("\n".join(pred_labels) + "\n")

    return {
        "acc": 100.0 * total_correct / max(total_frames, 1),
        "edit": float(np.mean(edit_scores)) if edit_scores else 0.0,
        "f1@10": float(np.mean(f1_10)) if f1_10 else 0.0,
        "f1@25": float(np.mean(f1_25)) if f1_25 else 0.0,
        "f1@50": float(np.mean(f1_50)) if f1_50 else 0.0,
        "frames": int(total_frames),
        "videos": len(loader.dataset),
    }

## 8. Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = []
best_metric = -1.0
best_path = CKPT_DIR / "best_model.pt"

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    start = time.time()

    train_losses = []
    train_accs = []

    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{NUM_EPOCHS}", leave=False)
    for video_ids, x, y, mask in pbar:
        x = x.to(device)
        y = y.to(device)
        mask = mask.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)
        loss = mstcn_loss(outputs, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        acc = frame_accuracy_from_logits(outputs[-1], y, mask)
        train_losses.append(float(loss.item()))
        train_accs.append(acc)

        pbar.set_postfix(loss=np.mean(train_losses), acc=100*np.mean(train_accs))

    train_metrics = evaluate_model(model, train_loader, save_predictions=False)
    test_metrics = evaluate_model(model, test_loader, save_predictions=False)

    row = {
        "epoch": epoch,
        "loss": float(np.mean(train_losses)),
        "batch_train_acc": 100.0 * float(np.mean(train_accs)),
        "train_acc": train_metrics["acc"],
        "train_edit": train_metrics["edit"],
        "train_f1@10": train_metrics["f1@10"],
        "train_f1@25": train_metrics["f1@25"],
        "train_f1@50": train_metrics["f1@50"],
        "test_acc": test_metrics["acc"],
        "test_edit": test_metrics["edit"],
        "test_f1@10": test_metrics["f1@10"],
        "test_f1@25": test_metrics["f1@25"],
        "test_f1@50": test_metrics["f1@50"],
        "seconds": time.time() - start,
    }

    history.append(row)
    print(row)

    # Use segmental F1@25 as model-selection metric.
    select_metric = row["test_f1@25"]
    if select_metric > best_metric:
        best_metric = select_metric
        torch.save({
            "model_state": model.state_dict(),
            "config": {
                "feature_dim": feature_dim,
                "num_classes": num_classes,
                "num_stages": NUM_STAGES,
                "num_layers": NUM_LAYERS,
                "num_f_maps": NUM_F_MAPS,
                "dropout": DROPOUT,
                "split_id": SPLIT_ID,
                "data_root": str(DATA_ROOT),
            },
            "epoch": epoch,
            "metrics": row,
            "id_to_label": id_to_label,
            "label_to_id": label_to_id,
        }, best_path)
        print("saved best:", best_path)

df_history = pd.DataFrame(history)
history_path = OUT_ROOT / "training_history.csv"
df_history.to_csv(history_path, index=False)

print("history saved:", history_path)
display(df_history.tail())

epoch 1/80:   0%|          | 0/46 [00:00<?, ?it/s]

{'epoch': 1, 'loss': 10.907141913538394, 'batch_train_acc': 9.274796195652174, 'train_acc': 11.429794520547945, 'train_edit': 14.089720726013184, 'train_f1@10': 12.535061969993478, 'train_f1@25': 4.554848880191346, 'train_f1@50': 0.6283974777125463, 'test_acc': 11.681547619047619, 'test_edit': 14.356575965881348, 'test_f1@10': 12.713529856387, 'test_f1@25': 3.8624338624338628, 'test_f1@50': 0.1322751322751323, 'seconds': 30.319361925125122}
saved best: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/mstcn_runs/mstcn_on_procedurevrl_breakfast_full_split1/checkpoints/best_model.pt


epoch 2/80:   0%|          | 0/46 [00:00<?, ?it/s]

{'epoch': 2, 'loss': 9.896815859753152, 'batch_train_acc': 13.533457880434781, 'train_acc': 16.65667808219178, 'train_edit': 24.34276580810547, 'train_f1@10': 28.004065949271432, 'train_f1@25': 24.932933884988678, 'train_f1@50': 11.402675710894888, 'test_acc': 17.73313492063492, 'test_edit': 26.300704956054688, 'test_f1@10': 30.281383079002126, 'test_f1@25': 28.166784714403764, 'test_f1@50': 13.006768363911222, 'seconds': 25.4072527885437}
saved best: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/mstcn_runs/mstcn_on_procedurevrl_breakfast_full_split1/checkpoints/best_model.pt


epoch 3/80:   0%|          | 0/46 [00:00<?, ?it/s]

...

epoch 79/80:   0%|          | 0/46 [00:00<?, ?it/s]

{'epoch': 79, 'loss': 5.685786464939946, 'batch_train_acc': 50.00764266304347, 'train_acc': 50.54366438356164, 'train_edit': 50.41645050048828, 'train_f1@10': 57.19892170234636, 'train_f1@25': 55.54910500458446, 'train_f1@50': 43.87617405425624, 'test_acc': 46.23015873015873, 'test_edit': 46.52887725830078, 'test_f1@10': 52.18674843674844, 'test_f1@25': 50.58828076685219, 'test_f1@50': 39.93596570977523, 'seconds': 24.332621335983276}


epoch 80/80:   0%|          | 0/46 [00:00<?, ?it/s]

{'epoch': 80, 'loss': 5.529996757921965, 'batch_train_acc': 51.20329483695652, 'train_acc': 51.25, 'train_edit': 51.689117431640625, 'train_f1@10': 58.597081571396636, 'train_f1@25': 56.658872520173894, 'train_f1@50': 45.13056806207491, 'test_acc': 47.048611111111114, 'test_edit': 47.50346374511719, 'test_f1@10': 53.00913372341943, 'test_f1@25': 51.535402428259566, 'test_f1@50': 40.28584730965683, 'seconds': 24.2597439289093}
history saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/mstcn_runs/mstcn_on_procedurevrl_breakfast_full_split1/training_history.csv


,epoch,loss,batch_train_acc,train_acc,train_edit,train_f1@10,train_f1@25,train_f1@50,test_acc,test_edit,test_f1@10,test_f1@25,test_f1@50,seconds
75,76,5.667698,50.309103,50.509418,50.733196,58.117457,55.649352,43.812060,47.395833,48.522453,55.505806,52.660880,42.033674,24.691229
76,77,5.567222,50.205503,53.882705,53.156883,60.174071,58.526326,46.839969,46.775794,47.153252,52.806594,50.832087,40.104774,24.510499
77,78,5.562507,50.554518,53.030822,52.410332,58.819961,56.988242,45.118642,45.362103,46.688240,50.282839,48.429318,39.024311,24.902006
78,79,5.685786,50.007643,50.543664,50.416451,57.198922,55.549105,43.876174,46.230159,46.528877,52.186748,50.588281,39.935966,24.332621
79,80,5.529997,51.203295,51.250000,51.689117,58.597082,56.658873,45.130568,47.048611,47.503464,53.009134,51.535402,40.285847,24.259744


## 9. Load best checkpoint and save final predictions

In [ ]:
checkpoint = torch.load(best_path, map_location=device)

best_model = MultiStageModel(
    num_stages=NUM_STAGES,
    num_layers=NUM_LAYERS,
    in_dim=feature_dim,
    num_f_maps=NUM_F_MAPS,
    num_classes=num_classes,
    dropout=DROPOUT,
).to(device)

best_model.load_state_dict(checkpoint["model_state"])

test_pred_dir = PRED_DIR / "test"
train_pred_dir = PRED_DIR / "train"

train_metrics = evaluate_model(best_model, train_loader, save_predictions=True, pred_dir=train_pred_dir)
test_metrics = evaluate_model(best_model, test_loader, save_predictions=True, pred_dir=test_pred_dir)

print("Best checkpoint epoch:", checkpoint["epoch"])
print("Train metrics:", train_metrics)
print("Test metrics:", test_metrics)

summary = {
    "run_name": RUN_NAME,
    "data_root": str(DATA_ROOT),
    "output_root": str(OUT_ROOT),
    "best_checkpoint": str(best_path),
    "best_epoch": int(checkpoint["epoch"]),
    "train_metrics": train_metrics,
    "test_metrics": test_metrics,
    "feature_dim": feature_dim,
    "feature_len": feature_len,
    "num_train_videos": len(train_ids),
    "num_test_videos": len(test_ids),
    "num_classes": num_classes,
    "note": "MS-TCN-style baseline on coarse ProcedureVRL clip-level features.",
}

summary_path = OUT_ROOT / "final_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

metrics_path = OUT_ROOT / "final_metrics.csv"
pd.DataFrame([
    {"split": "train", **train_metrics},
    {"split": "test", **test_metrics},
]).to_csv(metrics_path, index=False)

print("summary:", summary_path)
print("metrics:", metrics_path)
print(json.dumps(summary, indent=2))

Best checkpoint epoch: 69
Train metrics: {'acc': 51.81506849315068, 'edit': 52.22306442260742, 'f1@10': 59.32404999870754, 'f1@25': 57.35229021359159, 'f1@50': 44.95058670401137, 'frames': 23360, 'videos': 1460}
Test metrics: {'acc': 48.90873015873016, 'edit': 49.14902877807617, 'f1@10': 55.45678527821384, 'f1@25': 54.384011050677714, 'f1@50': 42.61379933998981, 'frames': 4032, 'videos': 252}
summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/mstcn_runs/mstcn_on_procedurevrl_breakfast_full_split1/final_summary.json
metrics: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/mstcn_runs/mstcn_on_procedurevrl_breakfast_full_split1/final_metrics.csv
{
  "run_name": "mstcn_on_procedurevrl_breakfast_full_split1",
  "data_root": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/breakfast/procedurevrl/runs/procedurevrl_breakfast_full_split1_split1_views16/mstcn_format",
  "output_root": "/content/drive/MyDrive/mmf_ta

## 10. Quick comparison with previous I3D/KD results table

In [ ]:
# Report-friendly comparison table.
# ProcedureVRL results below use coarse clip-level features with T=16.
# I3D results use frame-/segment-level features with variable temporal length T.
# Therefore this table is useful for progress reporting, but not for claiming a fully fair comparison.

def r2(x):
    return round(float(x), 2)

comparison_rows = [
    {
        "experiment": "Current experiment",
        "features": "ProcedureVRL coarse outputs",
        "model": "MS-TCN-style visual-only",
        "split": "Breakfast split 1",
        "feature_shape": f"[9871, {feature_len}]",
        "temporal_resolution": "16 clips per video",
        "acc": r2(test_metrics["acc"]),
        "edit": r2(test_metrics["edit"]),
        "f1@10": r2(test_metrics["f1@10"]),
        "f1@25": r2(test_metrics["f1@25"]),
        "f1@50": r2(test_metrics["f1@50"]),
        "directly_comparable_to_i3d": "No",
        "note": (
            "Successful ProcedureVRL -> TAS pipeline validation. "
            "Coarse clip-level outputs; not a fair frame-level comparison to I3D."
        ),
    },
    {
        "experiment": "Previous baseline",
        "features": "I3D",
        "model": "Official MS-TCN visual-only, 30 epochs",
        "split": "Breakfast split 1",
        "feature_shape": "[2048, T]",
        "temporal_resolution": "frame/segment-level variable T",
        "acc": 55.38,
        "edit": 44.97,
        "f1@10": 39.05,
        "f1@25": 34.80,
        "f1@50": 25.25,
        "directly_comparable_to_i3d": "Reference",
        "note": "Previous frame-level visual-only MS-TCN run.",
    },
    {
        "experiment": "Previous proof-of-concept",
        "features": "I3D + CLIP text teacher",
        "model": "Video-only student KD, lambda=0.05",
        "split": "Breakfast split 1",
        "feature_shape": "[2048, T]",
        "temporal_resolution": "frame/segment-level variable T",
        "acc": 70.98,
        "edit": 58.53,
        "f1@10": 50.16,
        "f1@25": 46.86,
        "f1@50": 38.33,
        "directly_comparable_to_i3d": "Yes, but text/visual spaces not aligned",
        "note": (
            "Previous KD proof-of-concept. "
            "Useful result, but visual I3D and CLIP text embeddings are not from the same latent space."
        ),
    },
]

df_comparison = pd.DataFrame(comparison_rows)

comparison_path = OUT_ROOT / "comparison_with_previous_runs.csv"
comparison_md_path = OUT_ROOT / "comparison_with_previous_runs.md"

df_comparison.to_csv(comparison_path, index=False)
comparison_md_path.write_text(df_comparison.to_markdown(index=False))

display(df_comparison)

print("saved CSV:", comparison_path)
print("saved Markdown:", comparison_md_path)

print("\nReport summary:")
print(
    f"ProcedureVRL coarse MS-TCN baseline on Breakfast split 1: "
    f"Acc={r2(test_metrics['acc'])}, "
    f"Edit={r2(test_metrics['edit'])}, "
    f"F1@10={r2(test_metrics['f1@10'])}, "
    f"F1@25={r2(test_metrics['f1@25'])}, "
    f"F1@50={r2(test_metrics['f1@50'])}."
)

print(
    "\nCaveat: these ProcedureVRL features have temporal length 16 per video, "
    "so this is a pipeline validation / preliminary baseline, not a fully fair "
    "frame-level comparison to I3D."
)